# Predicting Early Engagment - Logistic Regression
Tae Emmerson | Feb. 9 2026

In [22]:
import os
import re
import numpy as np
import pandas as pd
import snowflake.connector
import matplotlib.pyplot as plt
import seaborn as sns

from dotenv import load_dotenv

load_dotenv()

from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression # use optuna
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
import optuna

In [23]:
conn = snowflake.connector.connect(
    user=os.environ['SF_USERNAME'],
    password=os.environ['SF_PASSWORD'],   
    account=os.environ['SF_ACCOUNT'],
    warehouse="HN_WH",
    database="HN_DB",
    schema="PUBLIC",
)

df = pd.read_sql("""
    SELECT *
    FROM TRAIN_DATASET_V
""", conn)

/var/folders/y0/dmzwctj108g4w4xfxfj4cmhm0000gn/T/ipykernel_87216/1776946015.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("""


KeyboardInterrupt: 

In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4067 entries, 0 to 4066
Data columns (total 23 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   STORY_ID                4067 non-null   int64         
 1   CREATED_TS              4067 non-null   int64         
 2   CREATED_TS_UTC          4067 non-null   datetime64[us]
 3   AUTHOR                  4067 non-null   str           
 4   TITLE                   4067 non-null   str           
 5   URL                     3809 non-null   str           
 6   DOMAIN                  3809 non-null   str           
 7   SCORE_30M               4067 non-null   int64         
 8   COMMENTS_30M            4056 non-null   float64       
 9   KIDS_COUNT_30M          4067 non-null   int64         
 10  TITLE_LEN               4067 non-null   int64         
 11  IS_SHOW_HN              4067 non-null   int64         
 12  IS_ASK_HN               4067 non-null   int64         
 13 

Based on our findings from our exploratory data analysis `eda.ipynb`, there are three significant features:
1. `SCORE_30M` -> log transform
2. `COMMENTS_30M` -> log transform
3. `KIDS_COUNT_30m` -> log transform

Additionally, `TITLE` will be processed using a TF-IDF vectorizer. Future work can explore retrieving the actual article from the URL. 

Other features we'll include are `HOUR_UTC` and `DOW_UTC` which generally represent the hour and day of the week the post was made, as well as whether it is `IS_SHOW_HN` or `IS_ASK_HN`. A missingness indicator for `URL` will be added as well.

In [ ]:
df['log_SCORE_30M'] = np.log(df['SCORE_30M'] + 1)
df['log_COMMENTS_30M'] = np.log(df['COMMENTS_30M'] + 1)
df['log_KIDS_COUNT_30M'] = np.log(df['KIDS_COUNT_30M'] + 1)

df['URL_MISSING'] = df['URL'].isna().astype(int)

features = [
    'log_SCORE_30M', 'log_COMMENTS_30M', 'log_KIDS_COUNT_30M',
    'HOUR_UTC', 'DOW_UTC',
    'IS_SHOW_HN', 'IS_ASK_HN',
    'URL_MISSING',
    'TITLE'
]

To help our linear model, I'll be standardizing our features imputing with the median value for any missing values.

In [ ]:
tfidf = TfidfVectorizer(
        max_features=2000,
        ngram_range=(1, 2), # 1-grams and 2-grams
        min_df=5, # must occur in at least 5 titles (documents)
        max_df=0.50, # cannot occur in more than 70% of them
        lowercase=True,
        strip_accents='unicode'
    )

In [26]:
numeric_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
    ]
)

categorical_pipe = Pipeline(steps=[
    ('tfidf', TfidfVectorizer(
        max_features=2000,
        ngram_range=(1, 2), # 1-grams and 2-grams
        min_df=5, # must occur in at least 5 titles (documents)
        max_df=0.50, # cannot occur in more than 70% of them
        lowercase=True,
        strip_accents='unicode'
    ))
    ]
)


def create_pipeline(model):
    numeric_columns = features[:-1]
    preprocess = ColumnTransformer(transformers=[
        ('numeric', numeric_pipe, numeric_columns),
        ('text', categorical_pipe, 'TITLE')
    ])

    return Pipeline(steps=[
        ('preprocess', preprocess),
        ('model', model)
    ])

In [27]:
def evaluate(targets, probs, threshold=0.5):
    labels = probs > threshold
    return {
        "accuracy": accuracy_score(targets, labels),
        "precision": precision_score(targets, labels),
        "recall": recall_score(targets, labels),
        "auprc": average_precision_score(targets, probs),
        "auroc": roc_auc_score(targets, probs)
    }

In [28]:
X, y = df[features], df['Y_TOP30_DAILY']

# XGBoost

In [34]:
def objective(trial, X=X, y=y):

    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 200, 800),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1.0, 10.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 1.0),
    }

    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
        **params
    )

    pipe = create_pipeline(model)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    scores = []
    for fold, (tr, te) in enumerate(cv.split(X, y)):
        y_tr = y.iloc[tr]
        spw = (y_tr == 0).sum() / (y_tr == 1).sum()
        model.set_params(scale_pos_weight=spw)
        pipe.fit(X.iloc[tr], y.iloc[tr])
        # score on this fold (uses pipeline's score if set; otherwise do predict_proba + AP)
        from sklearn.metrics import average_precision_score
        proba = pipe.predict_proba(X.iloc[te])[:, 1]
        ap = average_precision_score(y.iloc[te], proba)
        scores.append(ap)

        trial.report(float(np.mean(scores)), step=fold)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(scores))

In [35]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(
    direction='maximize',
    study_name='logisitc_regression',
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=10)
)

study.optimize(objective, n_trials=500, n_jobs=6)

In [36]:
study.best_params

{'max_depth': 6,
 'min_child_weight': 1,
 'learning_rate': 0.010795615911226152,
 'n_estimators': 260,
 'subsample': 0.9484292827684672,
 'colsample_bytree': 0.9457358376855891,
 'gamma': 0.9145754841308361,
 'reg_lambda': 8.018682713322626,
 'reg_alpha': 0.2724282389727567}

In [37]:
study.best_value

0.34874700059686203

In [39]:
model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    n_jobs=-1,
    random_state=42,
    **study.best_params
)

pipe = create_pipeline(model)
pipe.fit(X, y)
evaluate(y, pipe.predict_proba(X)[:,1])

{'accuracy': 0.9724612736660929,
 'precision': 0.8387096774193549,
 'recall': 0.19548872180451127,
 'auprc': 0.45743252062760703,
 'auroc': 0.8966098520322159}